# Step 1.3.1 Validate Variable Candidates Data

# Check how the candidates are stored

It will show us the candidate table's schema, review a few records, and summarize how many candidates were found.

In [0]:

candidate_table = "dev_catalog.hrs_agent_metadata.hrs_variable_candidates"

if spark.catalog.tableExists(candidate_table):
    candidates = spark.table(candidate_table)

    print("Columns:")
    print(candidates.columns)

    print("Total candidate records:", candidates.count())
    
    display(candidates.orderBy('candidate_token','source_page').limit(10))
else:
    print(f"Candidate table not found: {candidate_table}")


# Verify the join between Pages and Candidates tables

It checks whether the candidate records can be matched to the full page text using the document ID, page number, and Codebook version.

In [0]:

from pyspark.sql import functions as F

candidates_table = "dev_catalog.hrs_agent_metadata.hrs_variable_candidates"
pages_table = "dev_catalog.hrs_agent_metadata.hrs_document_pages"

candidates = spark.table(candidates_table)
pages = spark.table(pages_table)

print("Candidate columns:", candidates.columns)
print("Document page columns:", pages.columns)

display(candidates.limit(3))
display(pages.limit(3))


# Run verification
This cell prints both schemas, identifies the shared join keys, and attempts to retrieve the complete Codebook page text for the three sample candidates.

### What we want to confirm
- The schemas show the actual column names and data types.
- The shared keys connect each candidate to the correct full page.
- The full page text provides enough context to review the variable's definition and construction.

In [0]:

from pyspark.sql import functions as F

candidates_table = (
    "dev_catalog.hrs_agent_metadata.hrs_variable_candidates"
)
pages_table = (
    "dev_catalog.hrs_agent_metadata.hrs_document_pages"
)

candidates = spark.table(candidates_table)
pages = spark.table(pages_table)

print("Candidate table schema:")
candidates.printSchema()

print("\nDocument pages table schema:")
pages.printSchema()

# Identify shared join keys without assuming their exact names.
preferred_keys = [
    "document_id",
    "page_number",
    "codebook_version"
]

join_keys = [
    key for key in preferred_keys
    if key in candidates.columns and key in pages.columns
]

print("\nShared join keys:", join_keys)

if not {"document_id", "page_number"}.issubset(join_keys):
    print(
        "STOP: document_id and page_number are not both "
        "shared keys. Review the schemas before joining."
    )
else:
    # Include Codebook version in the join when both tables have it.
    matched = candidates.alias("c").join(
        pages.alias("p"),
        on=join_keys,
        how="left"
    )

    display(
        matched.select(
            *[
                F.col(f"c.{key}").alias(key)
                for key in join_keys
            ],
            F.col("c.variable_name").alias("variable_name"),
            F.col("c.context_excerpt").alias("context_excerpt"),
            F.col("p.page_text").alias("full_page_text")
        )
        .filter(
            F.col("c.variable_name").isin(
                "R8LBPOSAFFECT6",
                "R8LBNEGAFFECT6",
                "R8LBPOSAFFECT"
            )
        )
        .limit(10)
    )


# Retrieve the full Codebook page text

### Interpret the results

Check these two things:

1. Join success: matched_page_number should be 2096, and full_page_text should contain the complete extracted text for that page.
2. Candidate coverage: confirm whether each of the three "candidate" tokens appears in the full text.
If the join succeeds, our next step will be to create a persistent review table containing the candidate token, full page text, Codebook version, review status, and reviewer notes.

We'll also make sure rerunning the extraction does not over write the decisions already made by you. 

Once that review workflow works, we can expand it to other "candidates" and automate metadata extraction while retaining source-page evidence.

In [0]:

from pyspark.sql import functions as F

candidates_table = (
    "dev_catalog.hrs_agent_metadata.hrs_variable_candidates"
)
pages_table = (
    "dev_catalog.hrs_agent_metadata.hrs_document_pages"
)

candidates = spark.table(candidates_table).alias("c")
pages = spark.table(pages_table).alias("p")

join_condition = (
    (F.col("c.document_id") == F.col("p.document_id"))
    & (F.col("c.source_page") == F.col("p.page_number"))
    & (
        F.col("c.codebook_version")
        == F.col("p.codebook_version")
    )
)

matched = candidates.join(
    pages,
    on=join_condition,
    how="left"
)

review_sample = (
    matched
    .filter(
        F.col("c.candidate_token").isin(
            "R8LBPOSAFFECT6",
            "R8LBNEGAFFECT6",
            "R8LBPOSAFFECT"
        )
        & (F.col("c.source_page") == 2096)
    )
    .select(
        F.col("c.document_id").alias("document_id"),
        F.col("c.source_page").alias("source_page"),
        F.col("c.candidate_token").alias("candidate_token"),
        F.col("c.codebook_version").alias("codebook_version"),
        F.col("c.context_excerpt").alias("context_excerpt"),
        F.col("p.page_number").alias("matched_page_number"),
        F.col("p.page_text").alias("full_page_text")
    )
)

print("Candidate rows:", review_sample.count())

display(review_sample)
